# Velas japonesas y redes convolucionales
## Notebook 01: descarga de datos y EDA
Trabajo final de Ciencia de Datos (A422), Universidad de San Andrés.

## Objetivo
Queremos ver si a partir de los patrones de velas japonesas se puede obtener una predicción útil para invertir con un rendimiento mayor al promedio del S&P 500.

Para eso vamos a armar imágenes con las velas de distintas acciones y entrenar redes convolucionales que, mirando solo esas imágenes, digan si la acción le va a ganar o no al S&P 500 en las semanas siguientes.

## Hipótesis
Partimos de que las velas por sí solas tienen poca información sobre el futuro del precio. Si fuera fácil predecirlo, cualquiera podría hacerlo. Por eso esperamos resultados apenas por encima del azar.

Aun así, creemos que algunas formas de velas, como las de reversión que aparecen en la literatura (por ejemplo el martillo), pueden darle al modelo una pequeña ventaja.

## Descarga de precios
Bajamos velas semanales de 2005 a 2025 de 50 acciones del S&P 500, 5 por cada uno de 10 sectores. Elegimos varios sectores para que el modelo no aprenda el comportamiento de una sola industria, y el período incluye crisis como la de 2008 y la de 2020 además de años de suba.

En cada sector mezclamos empresas a las que les fue muy bien con otras a las que les fue peor, para no quedarnos solo con las ganadoras que conocemos hoy.

Como referencia usamos SPY, el fondo que replica al S&P 500. Usamos precios ajustados por dividendos en todos los casos para que la comparación sea justa.

In [ ]:
!pip install -q yfinance
import yfinance as yf
import pandas as pd

# 5 acciones por sector, mezclando empresas a las que les fue bien y otras que no tanto
sectores = {
    "Tecnología": ["AAPL", "MSFT", "INTC", "CSCO", "IBM"],
    "Salud": ["JNJ", "PFE", "MRK", "UNH", "ABT"],
    "Finanzas": ["JPM", "BAC", "WFC", "C", "GS"],
    "Energía": ["XOM", "CVX", "COP", "SLB", "OXY"],
    "Consumo básico": ["KO", "PEP", "PG", "WMT", "CL"],
    "Consumo discrecional": ["AMZN", "HD", "MCD", "NKE", "SBUX"],
    "Industria": ["CAT", "BA", "GE", "HON", "MMM"],
    "Comunicaciones": ["T", "VZ", "DIS", "CMCSA", "NFLX"],
    "Servicios públicos": ["DUK", "SO", "NEE", "D", "AEP"],
    "Materiales": ["NEM", "FCX", "SHW", "ECL", "APD"],
}
tickers = [t for lista in sectores.values() for t in lista] + ["SPY"]

crudo = yf.download(tickers, start="2005-01-01", end="2025-12-31", interval="1wk",
                    auto_adjust=True, group_by="ticker", progress=False)

# paso a formato largo: una fila por acción y semana
filas = []
for t in tickers:
    df = crudo[t][["Open", "High", "Low", "Close"]].dropna().copy()
    df["ticker"] = t
    filas.append(df)
precios = pd.concat(filas).reset_index().rename(columns={"Date": "fecha"})
precios.columns = [c.lower() for c in precios.columns]

sector_de = {t: s for s, lista in sectores.items() for t in lista}
precios["sector"] = precios["ticker"].map(sector_de).fillna("Índice")

precios.to_csv("precios_semanales.csv", index=False)

print(precios.shape)
print("nulos:", precios.isna().sum().sum())
resumen = precios.groupby("ticker").agg(sector=("sector", "first"), semanas=("fecha", "count"),
                                        desde=("fecha", "min"), hasta=("fecha", "max"))
resumen.sort_values("semanas")

## Calidad de los datos
Bajaron las 51 series completas: 1096 semanas cada una, de enero de 2005 a diciembre de 2025, sin nulos.

Antes de seguir revisamos que los precios tengan sentido. Que el máximo de cada vela no quede por debajo de la apertura o el cierre, que el mínimo no quede por encima, que no haya precios en cero y si hay saltos semanales muy grandes que convenga mirar.

In [ ]:
p = precios.sort_values(["ticker", "fecha"]).copy()

# dejo un margen chico porque el ajuste por dividendos redondea los precios
mal_high = (p["high"] < p[["open", "close"]].max(axis=1) * 0.999).sum()
mal_low = (p["low"] > p[["open", "close"]].min(axis=1) * 1.001).sum()
print("velas con el máximo mal:", mal_high)
print("velas con el mínimo mal:", mal_low)
print("precios en cero o negativos:", (p[["open", "high", "low", "close"]] <= 0).sum().sum())

# semanas donde el cierre se movió más de 40%
p["var_semanal"] = p.groupby("ticker")["close"].pct_change()
saltos = p[p["var_semanal"].abs() > 0.4][["ticker", "sector", "fecha", "var_semanal"]]
print("semanas con saltos de más de 40%:", len(saltos))
saltos.sort_values("var_semanal")